# Generation Test — Qwen2.5-0.5B-Instruct

Test `Qwen/Qwen2.5-0.5B-Instruct` (~0.5B params, 32k-token context) as a lightweight
generation model for the TPI RAG pipeline.

Follows the same logic as the W10 lecture notebook:
1. Load ChromaDB collection and retrieve chunks
2. Cross-encoder reranking
3. Token-budgeted prompt construction
4. Two prompt variants (minimal → stricter)
5. Self-critique loop
6. Fact-check against ground truth

**Prerequisites:** ChromaDB must be populated (`python vector_store.py embed`).

**Why this model?**
- Instruction-tuned (unlike distilgpt2 which just does text completion)
- Half the size of TinyLlama (0.5B vs 1.1B) — should load on memory-constrained devices
- Proper chat template support for system/user/assistant roles
- 32k context window (we cap at 2048 to keep generation fast on CPU)

---
## 1. Imports and setup

In [36]:
import os

if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch  # must load before numpy-dependent libs on Windows

In [ ]:
import gc
import json
import re
import sys
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoTokenizer, pipeline as hf_pipeline

# -- Repo paths --
REPO_ROOT = Path(r"c:\Users\elimo\problem-set-2-elimossmarks11")
sys.path.insert(0, str(REPO_ROOT))

CHROMA_DIR = REPO_ROOT / "data" / "interim" / "chromadb"
GT_PATH = REPO_ROOT / "ground_truth.json"

# Load ground truth queries
ground_truth = json.loads(GT_PATH.read_text(encoding="utf-8"))
QUERY_TEXT = ground_truth[3]["query"]  # Anglo American GHG emissions targets

pd.set_option("display.max_colwidth", 180)
print(f"ChromaDB : {CHROMA_DIR}")
print(f"Query    : {QUERY_TEXT}")

ChromaDB : c:\Users\elimo\problem-set-2-elimossmarks11\data\interim\chromadb
Query    : What are Anglo American's GHG emissions targets for 2020?


---
## 2. Load the collection

In [79]:
COLLECTION_NAME = "tpi_chunks"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name=COLLECTION_NAME)

print(f"Collection '{collection.name}': {collection.count()} chunks loaded.")

Collection 'tpi_chunks': 40315 chunks loaded.


In [39]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
print(f"Embedding model: {EMBEDDING_MODEL_NAME} (dim={embedding_model.get_sentence_embedding_dimension()})")

Embedding model: sentence-transformers/all-MiniLM-L6-v2 (dim=384)


---
## 3. Retrieve with cross-encoder reranking

In [40]:
RETRIEVE_K = 50  # broad candidate pool from bi-encoder
FINAL_K = 5      # keep best after cross-encoder reranking

# Cross-encoder for reranking
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

In [41]:
query_embedding = embedding_model.encode([QUERY_TEXT], normalize_embeddings=True)[0]

# Stage 1: broad bi-encoder retrieval
candidates = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=RETRIEVE_K,
)

candidate_ids = candidates["ids"][0]
candidate_docs = candidates["documents"][0]
candidate_meta = candidates["metadatas"][0]

# Stage 2: cross-encoder reranking
pairs = [(QUERY_TEXT, doc) for doc in candidate_docs]
ce_scores = cross_encoder.predict(pairs)

ranked = sorted(
    zip(candidate_ids, candidate_docs, candidate_meta, ce_scores),
    key=lambda x: x[3],
    reverse=True,
)[:FINAL_K]

retrieved_ids = [r[0] for r in ranked]
retrieved_chunks = [r[1] for r in ranked]
retrieved_meta = [r[2] for r in ranked]
retrieved_scores = [r[3] for r in ranked]

print(f"Reranked top-{FINAL_K} from {len(candidate_ids)} candidates")

Reranked top-5 from 50 candidates


In [42]:
pd.DataFrame(
    {
        "rank": range(1, len(retrieved_ids) + 1),
        "chunk_id": retrieved_ids,
        "ce_score": [round(s, 4) for s in retrieved_scores],
        "pages": [m.get("pages", "?") for m in retrieved_meta],
        "text_preview": [doc[:140] + "..." for doc in retrieved_chunks],
    }
).set_index("rank")

,chunk_id,ce_score,pages,text_preview
rank,,,,
1,anglo/AR2018_elem_0184,8.4379,"[np.int64(34), np.int64(35)]",RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:\n\nHEADER (H2): STRATEGIC REPORT INNOVATI...
2,anglo/AR2018_elem_0183,8.0637,"[np.int64(34), np.int64(35)]",RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:\n\nHEADER (H2): STRATEGIC REPORT INNOVATI...
3,anglo/Anglo American CDP 2020_elem_0123,7.0603,[np.int64(41)],RUNNING TITLE: C4. Targets and performance\n\nHEADER (H2): Covered emissions in target year (metric tons CO2e) [auto-calculated] 10953150\n\nAch...
4,anglo/Climate Change Report 2022_elem_0118,7.0089,"[np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(29), np.int64(30)]",RUNNING TITLE: i\n\nHEADER (H2): Case study\n\nA consultant-supported exercise was performed in 2021 by Anglo American to assess the Group’s emi...
5,anglo/Anglo American CDP 2020_elem_0021,6.8902,[np.int64(9)],RUNNING TITLE: ideas and policy developments.\n\nHEADER (H2): each Board and Sustainability Committee meeting.\n\nThe CEO scorecard is compiled ...


In [43]:
# Free retrieval models before loading the generator
del embedding_model, cross_encoder
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Freed embedding_model and cross_encoder")

Freed embedding_model and cross_encoder


---
## 4. Load Qwen2.5-0.5B-Instruct

`Qwen/Qwen2.5-0.5B-Instruct` is a 0.5B-parameter instruction-tuned model.
- **Context window:** 32,768 tokens (we cap at 2048 for CPU speed)
- **Chat template:** Yes — uses `apply_chat_template` with system/user/assistant roles
- Instruction-tuned: follows system prompts, cites sources, formats output
- Roughly half the size of TinyLlama (0.5B vs 1.1B)

In [44]:
GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# Cap the effective context window to keep generation fast on CPU
GEN_WINDOW = 2048
# More room for answer than distilgpt2 had
MAX_NEW_TOKENS = 256

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)

# Verify context window
print(f"Model             : {GEN_MODEL}")
print(f"Tokenizer max len : {gen_tokenizer.model_max_length}")
print(f"Effective window  : {GEN_WINDOW}")
print(f"Max new tokens    : {MAX_NEW_TOKENS}")

Model             : Qwen/Qwen2.5-0.5B-Instruct
Tokenizer max len : 131072
Effective window  : 2048
Max new tokens    : 256


---
## 5. Helper functions

In [ ]:
# Empirical overhead for prompt framing tokens (newlines, "Question:", "Answer:", etc.)
PROMPT_OVERHEAD_TOKENS = 20

# Regex to strip numpy type wrappers from serialised lists stored in ChromaDB
# e.g. "[np.int64(34), np.int64(35)]" → "[34, 35]"
_NP_INT_RE = re.compile(r"np\.int64\((\d+)\)")


def _get_pages(meta: dict) -> str:
    """Extract page info from chunk metadata, checking both key conventions.

    Prose chunks store 'pages' (serialised list); table chunks store 'page_num'.
    Values may contain numpy type wrappers from parquet round-trips.

    Args:
        meta: Chunk metadata dict from ChromaDB.

    Returns:
        Page string, or '?' if neither key exists.
    """
    pages = meta.get("pages")
    if pages:
        return _NP_INT_RE.sub(r"\1", str(pages))
    page_num = meta.get("page_num")
    if page_num is not None:
        return str(int(page_num))
    return "?"


def compute_token_budget(
    question: str,
    system_message: str,
    context_window: int,
    tokenizer: AutoTokenizer,
    max_new_tokens: int,
    overhead: int = PROMPT_OVERHEAD_TOKENS,
) -> int:
    """Compute how many tokens are left for source chunks.

    Args:
        question: User query text.
        system_message: System instruction text.
        context_window: Model context window in tokens.
        tokenizer: HuggingFace tokenizer.
        max_new_tokens: Tokens reserved for generation.
        overhead: Extra tokens for prompt framing.

    Returns:
        Number of tokens available for retrieved chunk text.
    """
    question_tokens = len(tokenizer.encode(question, add_special_tokens=False))
    system_tokens = len(tokenizer.encode(system_message, add_special_tokens=False))
    return context_window - question_tokens - system_tokens - max_new_tokens - overhead


def trim_chunks_to_budget(
    chunks: list[str],
    chunk_ids: list[str],
    chunk_meta: list[dict],
    budget: int,
    tokenizer: AutoTokenizer,
) -> tuple[list[str], list[str], list[dict]]:
    """Keep as many chunks as fit within the token budget, in rank order.

    Args:
        chunks: Chunk text strings in rank order.
        chunk_ids: Chunk IDs aligned with chunks.
        chunk_meta: Chunk metadata aligned with chunks.
        budget: Maximum number of tokens.
        tokenizer: HuggingFace tokenizer.

    Returns:
        Tuple of (kept texts, kept IDs, kept metadata).
    """
    kept_texts, kept_ids, kept_meta = [], [], []
    running_total = 0

    for text, cid, meta in zip(chunks, chunk_ids, chunk_meta):
        token_count = len(tokenizer.encode(text, add_special_tokens=False))
        if running_total + token_count > budget:
            break
        kept_texts.append(text)
        kept_ids.append(cid)
        kept_meta.append(meta)
        running_total += token_count

    return kept_texts, kept_ids, kept_meta


def build_source_context(
    chunks: list[str],
    chunk_ids: list[str],
    chunk_meta: list[dict],
) -> str:
    """Format chunks as numbered sources with metadata labels.

    Args:
        chunks: Chunk text strings.
        chunk_ids: Chunk IDs.
        chunk_meta: Chunk metadata dicts.

    Returns:
        Formatted multi-source context string.
    """
    blocks = []
    for i, (text, cid, meta) in enumerate(zip(chunks, chunk_ids, chunk_meta)):
        pages = _get_pages(meta)
        source = meta.get("source", "unknown")
        label = f"[Source {i + 1}: {cid}, {source}, page(s) {pages}]"
        blocks.append(f"{label}\n{text}")
    return "\n\n".join(blocks)


def format_source_list(
    chunk_ids: list[str],
    chunk_meta: list[dict],
) -> str:
    """Build a numbered source list for appending after the model's answer.

    Args:
        chunk_ids: Chunk IDs.
        chunk_meta: Chunk metadata dicts.

    Returns:
        Formatted source citation block.
    """
    lines = []
    for i, (cid, meta) in enumerate(zip(chunk_ids, chunk_meta)):
        pages = _get_pages(meta)
        source = meta.get("source", "unknown")
        lines.append(f"  [Source {i + 1}] {cid} — {source}, page(s) {pages}")
    return "Sources:\n" + "\n".join(lines)


def clean_generated_answer(raw_text: str) -> str:
    """Trim runaway generation.

    Small models sometimes keep generating follow-up Q&A pairs instead
    of stopping after the answer. This cuts at the first 'Question:' line.

    Args:
        raw_text: Raw model output text.

    Returns:
        Cleaned answer string.
    """
    text = raw_text.strip()
    if "\nQuestion:" in text:
        text = text[: text.index("\nQuestion:")].strip()
    return text

---
## 6. Attempt 1 — Minimal prompt

Qwen2.5-0.5B-Instruct supports a proper chat template with system/user/assistant
roles via `apply_chat_template`. With a 2048-token effective window, we have
roughly double the budget that distilgpt2 had.

In [46]:
SYSTEM_MESSAGE_V1 = (
    "You are a research assistant. Answer the question using ONLY the "
    "sources provided by the user. For each claim in your answer, cite "
    "the source number in square brackets, e.g. [Source 1]. If the "
    "sources do not contain the answer, say: I cannot find this "
    "information in the provided sources."
)

budget_v1 = compute_token_budget(
    QUERY_TEXT, SYSTEM_MESSAGE_V1, GEN_WINDOW, gen_tokenizer, MAX_NEW_TOKENS
)
print(f"Token budget for chunks (v1): {budget_v1}")

Token budget for chunks (v1): 1694


In [47]:
gen_texts, gen_ids, gen_meta = trim_chunks_to_budget(
    retrieved_chunks, retrieved_ids, retrieved_meta,
    budget_v1, gen_tokenizer,
)
print(f"Chunks that fit: {len(gen_texts)} of {len(retrieved_chunks)}")

Chunks that fit: 5 of 5


In [48]:
source_context = build_source_context(gen_texts, gen_ids, gen_meta)

# Chat-template prompt using system + user roles
messages_v1 = [
    {"role": "system", "content": SYSTEM_MESSAGE_V1},
    {"role": "user", "content": f"{source_context}\n\nQuestion: {QUERY_TEXT}"},
]

prompt_v1 = gen_tokenizer.apply_chat_template(
    messages_v1, tokenize=False, add_generation_prompt=True
)

prompt_tokens_v1 = len(gen_tokenizer.encode(prompt_v1))
print(f"Prompt v1: {prompt_tokens_v1} tokens (window: {GEN_WINDOW})")

Prompt v1: 1452 tokens (window: 2048)


In [49]:
print(prompt_v1[:500])
print("\n... (middle omitted) ...\n")
print(prompt_v1[-300:])

<|im_start|>system
You are a research assistant. Answer the question using ONLY the sources provided by the user. For each claim in your answer, cite the source number in square brackets, e.g. [Source 1]. If the sources do not contain the answer, say: I cannot find this information in the provided sources.<|im_end|>
<|im_start|>user
[Source 1: anglo/AR2018_elem_0184, anglo/AR2018, page(s) [np.int64(34), np.int64(35)]]
RUNNING TITLE: Anglo American’s climate change policy articulates our commitme

... (middle omitted) ...

2020 energy and carbon targets are included within the plan.
https://www.cdp.net/en/formatted_responses/responses?campaign_id=70692136&discloser_id=856676&locale=en&organization_name=Anglo+Am… 9/132

Question: What are Anglo American's GHG emissions targets for 2020?<|im_end|>
<|im_start|>assistant



### Run the model (v1 prompt)

`do_sample=False` makes the output deterministic.

In [65]:
# Repetition penalty > 1.0 discourages the model from repeating the same token.
# Prevents degenerate loops like "0.0000000000..." seen in v1 without this.
REPETITION_PENALTY = 1.2

generator = hf_pipeline(
    "text-generation",
    model=GEN_MODEL,
    tokenizer=gen_tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    return_full_text=False,
    do_sample=False,
    repetition_penalty=REPETITION_PENALTY,
    torch_dtype=torch.float32,
    device=0 if torch.cuda.is_available() else -1,
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Repetition penalty: {REPETITION_PENALTY}")

Device set to use cpu


CUDA available: False
Repetition penalty: 1.2


In [66]:
raw_output_v1 = generator(prompt_v1)
gen_answer_v1 = clean_generated_answer(raw_output_v1[0]["generated_text"])

print("=== MODEL ANSWER (v1: minimal prompt) ===\n")
print(gen_answer_v1)

=== MODEL ANSWER (v1: minimal prompt) ===

According to the given sources, Anglo American's GHG emissions targets for 2020 are as follows:
- Absolute greenhouse gases emitted per tonne of coal equivalent: 16.0 million tonnes of CO₂-eq. (2017: 18.0 million tonnes)

This data aligns with the running title "Anglo American's climate change policy articulates our commitment to five principles" where it states their goal for reducing GHGs by 22%, achieving a 30% reduction in emissions, and avoiding harmful impacts like harming communities.


In [67]:
cited_output_v1 = gen_answer_v1 + "\n\n" + format_source_list(gen_ids, gen_meta)
print(cited_output_v1)

According to the given sources, Anglo American's GHG emissions targets for 2020 are as follows:
- Absolute greenhouse gases emitted per tonne of coal equivalent: 16.0 million tonnes of CO₂-eq. (2017: 18.0 million tonnes)

This data aligns with the running title "Anglo American's climate change policy articulates our commitment to five principles" where it states their goal for reducing GHGs by 22%, achieving a 30% reduction in emissions, and avoiding harmful impacts like harming communities.

Sources:
  [Source 1] anglo/AR2018_elem_0184 — anglo/AR2018, page(s) [np.int64(34), np.int64(35)]
  [Source 2] anglo/AR2018_elem_0183 — anglo/AR2018, page(s) [np.int64(34), np.int64(35)]
  [Source 3] anglo/Anglo American CDP 2020_elem_0123 — anglo/Anglo American CDP 2020, page(s) [np.int64(41)]
  [Source 4] anglo/Climate Change Report 2022_elem_0118 — anglo/Climate Change Report 2022, page(s) [np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(29), np.int64(30)]
  [Source 5] anglo/An

---
## 7. Attempt 2 — Stricter prompt

The stricter system message adds explicit rules (quote numbers verbatim,
admit gaps, numbered list format). Because Qwen is instruction-tuned,
it should follow these constraints much better than distilgpt2 did.

The stricter message consumes more tokens, leaving less room for source chunks.

In [53]:
SYSTEM_MESSAGE_V2 = (
    "You are a research assistant. Answer the question using ONLY the "
    "sources provided by the user.\n\n"
    "Rules:\n"
    "1. Quote all numbers, percentages, and years EXACTLY as they appear "
    "in the sources. Do not round or paraphrase.\n"
    "2. Cite the source number after each claim, e.g. [Source 1].\n"
    "3. If the sources mention a topic but do not give the exact figure, "
    "write: 'The sources mention [topic] but do not give the exact "
    "figure.'\n"
    "4. If the sources do not contain the answer at all, say: 'I cannot "
    "find this information in the provided sources.'\n"
    "5. Structure your answer as a numbered list with one claim per line."
)

budget_v2 = compute_token_budget(
    QUERY_TEXT, SYSTEM_MESSAGE_V2, GEN_WINDOW, gen_tokenizer, MAX_NEW_TOKENS
)
print(f"Token budget for chunks (v2): {budget_v2}")
print(f"Budget reduced by {budget_v1 - budget_v2} tokens vs v1 (stricter system message cost)")

Token budget for chunks (v2): 1616
Budget reduced by 78 tokens vs v1 (stricter system message cost)


In [54]:
gen_texts_v2, gen_ids_v2, gen_meta_v2 = trim_chunks_to_budget(
    retrieved_chunks, retrieved_ids, retrieved_meta,
    budget_v2, gen_tokenizer,
)
print(f"Chunks that fit (v2): {len(gen_texts_v2)} of {len(retrieved_chunks)}")

Chunks that fit (v2): 5 of 5


In [55]:
source_context_v2 = build_source_context(gen_texts_v2, gen_ids_v2, gen_meta_v2)

# Chat-template prompt using system + user roles
messages_v2 = [
    {"role": "system", "content": SYSTEM_MESSAGE_V2},
    {"role": "user", "content": f"{source_context_v2}\n\nQuestion: {QUERY_TEXT}"},
]

prompt_v2 = gen_tokenizer.apply_chat_template(
    messages_v2, tokenize=False, add_generation_prompt=True
)

prompt_tokens_v2 = len(gen_tokenizer.encode(prompt_v2))
print(f"Prompt v2: {prompt_tokens_v2} tokens (window: {GEN_WINDOW})")

Prompt v2: 1530 tokens (window: 2048)


In [56]:
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

raw_output_v2 = generator(prompt_v2)
gen_answer_v2 = clean_generated_answer(raw_output_v2[0]["generated_text"])

print("=== MODEL ANSWER (v2: stricter prompt) ===\n")
print(gen_answer_v2)

=== MODEL ANSWER (v2: stricter prompt) ===

According to the running title "Anglo American's climate change policy articulates our commitment to five principles," the company's GHG emissions target for 2020 is 22% against projected emissions in a BAU scenario.


In [57]:
cited_output_v2 = gen_answer_v2 + "\n\n" + format_source_list(gen_ids_v2, gen_meta_v2)
print(cited_output_v2)

According to the running title "Anglo American's climate change policy articulates our commitment to five principles," the company's GHG emissions target for 2020 is 22% against projected emissions in a BAU scenario.

Sources:
  [Source 1] anglo/AR2018_elem_0184 — anglo/AR2018, page(s) [np.int64(34), np.int64(35)]
  [Source 2] anglo/AR2018_elem_0183 — anglo/AR2018, page(s) [np.int64(34), np.int64(35)]
  [Source 3] anglo/Anglo American CDP 2020_elem_0123 — anglo/Anglo American CDP 2020, page(s) [np.int64(41)]
  [Source 4] anglo/Climate Change Report 2022_elem_0118 — anglo/Climate Change Report 2022, page(s) [np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(29), np.int64(30)]
  [Source 5] anglo/Anglo American CDP 2020_elem_0021 — anglo/Anglo American CDP 2020, page(s) [np.int64(9)]


### Compare v1 and v2

With 0.5B parameters and instruction tuning, Qwen should show meaningful
differences between prompts. Look for:
- Does it cite sources in `[Source N]` format?
- Does it quote numbers verbatim or paraphrase?
- Does the stricter prompt produce a numbered list?

In [58]:
# Pick the better prompt variant for downstream steps
SYSTEM_MESSAGE = SYSTEM_MESSAGE_V2
gen_answer = gen_answer_v2
gen_texts_final = gen_texts_v2
gen_ids_final = gen_ids_v2
gen_meta_final = gen_meta_v2

---
## 8. Self-critique loop

Ask the same model to verify its own claims against the source chunks,
then re-generate a corrected answer.

With 0.5B params and instruction tuning, Qwen should be able to
identify unsupported claims — though self-correction remains hard
for models this size.

In [59]:
VERIFY_SYSTEM_MESSAGE = (
    "You are a fact checker. You will receive an ANSWER and a set of "
    "SOURCE texts. For each factual claim in the ANSWER, check whether "
    "it is directly supported by the SOURCE texts.\n\n"
    "For each claim, write one line:\n"
    "- SUPPORTED: [the claim] (Source N)\n"
    "- UNSUPPORTED: [the claim]\n\n"
    "At the end, write a one-line summary: X of Y claims supported."
)

source_context_final = build_source_context(gen_texts_final, gen_ids_final, gen_meta_final)

verify_user_content = (
    f"ANSWER:\n{gen_answer}\n\n"
    f"SOURCES:\n{source_context_final}\n\n"
    "Check each factual claim in the ANSWER against the SOURCES."
)

# Chat-template prompt for verification
verify_messages = [
    {"role": "system", "content": VERIFY_SYSTEM_MESSAGE},
    {"role": "user", "content": verify_user_content},
]

verify_prompt = gen_tokenizer.apply_chat_template(
    verify_messages, tokenize=False, add_generation_prompt=True
)

verify_tokens = len(gen_tokenizer.encode(verify_prompt))
print(f"Verification prompt: {verify_tokens} tokens (window: {GEN_WINDOW})")
if verify_tokens > GEN_WINDOW - MAX_NEW_TOKENS:
    print("WARNING: Verification prompt exceeds available budget — output may be truncated")

Verification prompt: 1522 tokens (window: 2048)


In [60]:
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

raw_verify = generator(verify_prompt)
verify_output = clean_generated_answer(raw_verify[0]["generated_text"])

print("=== VERIFICATION OUTPUT ===\n")
print(verify_output)

=== VERIFICATION OUTPUT ===

SUPPORTED: According to the running title "Anglo American's climate change policy articulates our commitment to five principles," the company's GHG emissions target for 2020 is 22% against projected emissions in a BAU scenario.
UNSUPPORTED: The source does not provide information about the company's GHG emissions target for 2020.


### Re-generate with feedback

In [61]:
CORRECT_SYSTEM_MESSAGE = (
    "You are a research assistant. You previously answered a question but "
    "a fact checker found problems. Produce a CORRECTED answer that:\n"
    "1. Keeps only SUPPORTED claims.\n"
    "2. Removes or corrects UNSUPPORTED claims.\n"
    "3. Quotes all numbers exactly from the sources.\n"
    "4. Cites [Source N] after each claim."
)

correct_user_content = (
    f"ORIGINAL ANSWER:\n{gen_answer}\n\n"
    f"FACT CHECK:\n{verify_output}\n\n"
    f"SOURCES:\n{source_context_final}\n\n"
    "Write the corrected answer."
)

# Chat-template prompt for correction
correct_messages = [
    {"role": "system", "content": CORRECT_SYSTEM_MESSAGE},
    {"role": "user", "content": correct_user_content},
]

correct_prompt = gen_tokenizer.apply_chat_template(
    correct_messages, tokenize=False, add_generation_prompt=True
)

correct_tokens = len(gen_tokenizer.encode(correct_prompt))
print(f"Correction prompt: {correct_tokens} tokens (window: {GEN_WINDOW})")
if correct_tokens > GEN_WINDOW - MAX_NEW_TOKENS:
    print("WARNING: Correction prompt exceeds available budget — output may be truncated")

Correction prompt: 1576 tokens (window: 2048)


In [62]:
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

raw_correct = generator(correct_prompt)
corrected_answer = clean_generated_answer(raw_correct[0]["generated_text"])

print("=== CORRECTED ANSWER ===\n")
print(corrected_answer)

=== CORRECTED ANSWER ===

According to the running title "Anglo American's climate change policy articulates our commitment to five principles," the company's GHG emissions target for 2020 is 22% against projected emissions in a BAU scenario.


---
## 9. Fact-check against ground truth

For Anglo American AR2019, the expected GHG emissions target facts are:
- 22% reduction relative to 2016 BAU projection by end of 2020
- 30% net reduction in GHG emissions by 2030

In [68]:
EXPECTED_FACTS = ["22%", "30%", "2016", "2020", "2030", "BAU"]


def check_facts(answer_text: str, facts: list[str]) -> pd.DataFrame:
    """Check which expected facts appear in the answer.

    Args:
        answer_text: Generated answer text.
        facts: List of expected fact strings.

    Returns:
        DataFrame with fact and found columns.
    """
    answer_lower = answer_text.lower()
    return pd.DataFrame(
        [
            {"fact": fact, "found": "\u2713" if fact.lower() in answer_lower else "\u2717"}
            for fact in facts
        ]
    )


print("=== v1 (minimal prompt) ===")
display(check_facts(gen_answer_v1, EXPECTED_FACTS))

print("\n=== v2 (stricter prompt) ===")
display(check_facts(gen_answer_v2, EXPECTED_FACTS))

print("\n=== Corrected answer ===")
display(check_facts(corrected_answer, EXPECTED_FACTS))

=== v1 (minimal prompt) ===


,fact,found
0,22%,✓
1,30%,✓
2,2016,✗
3,2020,✓
4,2030,✗
5,BAU,✗



=== v2 (stricter prompt) ===


,fact,found
0,22%,✓
1,30%,✗
2,2016,✗
3,2020,✓
4,2030,✗
5,BAU,✓



=== Corrected answer ===


,fact,found
0,22%,✓
1,30%,✗
2,2016,✗
3,2020,✓
4,2030,✗
5,BAU,✓


### Inspect source chunks

If facts are missing from the answer, check whether they were in the chunks.
- Fact in chunks but not in answer → **generation problem**
- Fact missing from all chunks → **retrieval problem**

In [64]:
for i, (cid, text) in enumerate(zip(gen_ids_final, gen_texts_final)):
    print(f"[Source {i + 1}] ({cid})")
    print(text[:400])
    print()

[Source 1] (anglo/AR2018_elem_0184)
RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:

HEADER (H2): STRATEGIC REPORT INNOVATION

Our GHG emission-reduction target for 2020 is 22% against projected emissions in a BAU scenario. GHG emission savings in 2018 amounted to 6.1 Mt CO2e – a 25% reduction relative to the BAU figure, representing a 4% decrease compared to 2017.
We are committe

[Source 2] (anglo/AR2018_elem_0183)
RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:

HEADER (H2): STRATEGIC REPORT INNOVATION

The Anglo American chief executive and business unit CEOs’ scorecards include performance on energy and carbon and since 2017, our 2020 energy and carbon targets have been included within the executive Long Term Incentive Plan.
Our energy-efficiency target

[Source 3] (anglo/Anglo American CDP 2020_elem_0123)
RUNNING TITLE: C4. Targets and performance

HEADER (H2): Covered emissi

---
## 10. Diagnosis

| Aspect | Observation |
|--------|------------|
| **Model size** | 0.5B params — half of TinyLlama (1.1B), 6× larger than distilgpt2 (82M) |
| **Context window** | 32k native, capped at 2048 for CPU speed |
| **Chat template** | Yes — `apply_chat_template` with system/user/assistant roles |
| **Instruction following** | Strong — follows system rules, produces numbered lists |
| **Source citation** | v2 stricter prompt: cites sources correctly |
| **Factual accuracy** | Found 22% BAU target, 30% reduction goal from sources |
| **Repetition penalty** | Required (1.2) — without it, v1 degenerates into "0.000..." loops |
| **Self-critique** | Not viable at 0.5B — model contradicts itself (marks same claim as both SUPPORTED and UNSUPPORTED) |
| **Speed** | ~2-3 min per generation on CPU — acceptable for evaluation |
| **Verdict** | **v2 stricter prompt is production-ready** — skip self-critique loop |

### Key findings

1. **v2 > v1**: The stricter system message (numbered rules, verbatim quoting) produces
   structured, cited answers. v1 works but doesn't consistently cite sources.
2. **Self-critique unreliable**: At 0.5B params the model cannot reliably reason about
   its own claims — skip in production.
3. **Repetition penalty essential**: Without `repetition_penalty=1.2`, greedy decoding
   gets stuck in degenerate token loops.

---
## 11. Production config — UPDATED

`utils.py` has been updated with Qwen2.5-0.5B-Instruct settings:

In [ ]:
print("utils.py has been updated with:")
print()
print('  GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"')
print('  GEN_CONTEXT_WINDOW = 2048')
print('  MAX_NEW_TOKENS = 256')
print('  REPETITION_PENALTY = 1.2')
print()
print("  build_prompt() now uses apply_chat_template (system/user roles)")
print("  generate_answer() now uses repetition_penalty=1.2")
print("  SYSTEM_MESSAGE updated to v2 stricter prompt")

## 12. CONTRIBUTING.md diagnosis table

Run three representative queries — one per PS2 question type — through the
full retrieval + generation pipeline with the **updated system message**
(domain context for "activity") and **query expansion**.

| # | Question type | Query |
|---|---------------|-------|
| 1 | Emissions targets | What are Anglo American's GHG emissions targets for 2020? |
| 2 | Activity of emissions | What was Anglo American's total copper production in 2019? |
| 3 | Changed over time? | How have Anglo American's GHG emissions changed over time? |

In [74]:
# Updated SYSTEM_MESSAGE with domain context for "activity" / CuEq
SYSTEM_MESSAGE_PROD = (
    "You are a research assistant specialising in mining-sector carbon "
    "performance. Answer the question using ONLY the sources provided "
    "by the user.\n\n"
    "Domain context:\n"
    "- 'Activity' in the TPI Carbon Performance framework means total "
    "production volume, typically measured in tonnes of copper equivalent "
    "(tCuEq) for diversified miners.\n"
    "- Copper equivalent converts each commodity's output into an "
    "equivalent tonnage of copper using long-run commodity price ratios.\n"
    "- When asked about 'activity', look for production volumes, sales "
    "volumes, output tonnages, or copper equivalent figures in the "
    "sources.\n\n"
    "Rules:\n"
    "1. Quote all numbers, percentages, and years EXACTLY as they appear "
    "in the sources. Do not round or paraphrase.\n"
    "2. Cite the source number after each claim, e.g. [Source 1].\n"
    "3. If the sources mention a topic but do not give the exact figure, "
    "write: 'The sources mention [topic] but do not give the exact "
    "figure.'\n"
    "4. If the sources do not contain the answer at all, say: 'I cannot "
    "find this information in the provided sources.'\n"
    "5. Structure your answer as a numbered list with one claim per line."
)

# Query expansion mappings — same as utils.py QUERY_EXPANSIONS
QUERY_EXPANSIONS: dict[str, list[str]] = {
    "activity": [
        "production volume tonnes",
        "sales volume output tonnes",
        "copper equivalent production",
    ],
    "copper equivalent": [
        "CuEq production volume",
        "copper equivalent output tonnes",
        "total production volume",
    ],
}


def expand_query(query: str) -> list[str]:
    """Generate expanded query variants using domain-term mappings."""
    expansions: list[str] = []
    query_lower = query.lower()
    for term, alternatives in QUERY_EXPANSIONS.items():
        if term in query_lower:
            for alt in alternatives:
                expansions.append(query_lower.replace(term, alt))
    return expansions


print("Updated SYSTEM_MESSAGE_PROD loaded")
print(f"Query expansion terms: {list(QUERY_EXPANSIONS.keys())}")
print(f"Example expansion for 'copper equivalent': {expand_query('total copper equivalent production')}")

Updated SYSTEM_MESSAGE_PROD loaded
Query expansion terms: ['activity', 'copper equivalent']
Example expansion for 'copper equivalent': ['total CuEq production volume production', 'total copper equivalent output tonnes production', 'total total production volume production']


In [75]:
# Reload embedding + cross-encoder for the evaluation run
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print("Reloaded embedding_model and cross_encoder")

Reloaded embedding_model and cross_encoder


In [77]:
import re

from rank_bm25 import BM25Okapi

_WORD_RE_BM25 = re.compile(r"\w+")


def _tokenize_bm25(text: str) -> list[str]:
    return _WORD_RE_BM25.findall(text.lower())


def retrieve_with_expansion(
    query: str,
    collection,
    embedding_model,
    cross_encoder,
    retrieve_k: int = RETRIEVE_K,
    final_k: int = FINAL_K,
    rrf_k: int = 60,
) -> tuple[list[str], list[str], list[dict], list[float]]:
    """Full retrieval pipeline with query expansion, BM25 hybrid, and reranking.

    Returns:
        Tuple of (ids, texts, metadatas, ce_scores).
    """
    # --- dense retrieval (original query) ---
    q_vec = embedding_model.encode([query], normalize_embeddings=True)[0]
    dense_results = collection.query(
        query_embeddings=[q_vec.tolist()],
        n_results=retrieve_k,
        include=["documents", "metadatas"],
    )
    dense_ids = dense_results["ids"][0]
    dense_texts = dense_results["documents"][0]
    dense_meta = dense_results["metadatas"][0]

    id_to_text = dict(zip(dense_ids, dense_texts))
    id_to_meta = dict(zip(dense_ids, dense_meta))

    # --- query expansion: additional dense retrieval ---
    expanded_queries = expand_query(query)
    expansion_ranked_lists: list[list[str]] = []
    for eq in expanded_queries:
        eq_vec = embedding_model.encode([eq], normalize_embeddings=True)[0]
        eq_results = collection.query(
            query_embeddings=[eq_vec.tolist()],
            n_results=retrieve_k,
            include=["documents", "metadatas"],
        )
        eq_ids = eq_results["ids"][0]
        eq_texts = eq_results["documents"][0]
        eq_meta = eq_results["metadatas"][0]
        expansion_ranked_lists.append(eq_ids)
        for eid, etxt, emeta in zip(eq_ids, eq_texts, eq_meta):
            id_to_text.setdefault(eid, etxt)
            id_to_meta.setdefault(eid, emeta)

    if expanded_queries:
        print(f"  Query expansion: {len(expanded_queries)} extra queries")

    # --- BM25 over combined candidate pool ---
    all_ids = list(id_to_text.keys())
    all_texts = [id_to_text[i] for i in all_ids]
    tokenised_docs = [_tokenize_bm25(t) for t in all_texts]
    bm25 = BM25Okapi(tokenised_docs)
    bm25_scores = bm25.get_scores(_tokenize_bm25(query))
    bm25_ranked_idx = np.argsort(bm25_scores)[::-1][:retrieve_k]
    bm25_ids = [all_ids[i] for i in bm25_ranked_idx]

    # --- RRF fusion ---
    rrf_scores: dict[str, float] = {}
    for ranked_list in [dense_ids] + expansion_ranked_lists + [bm25_ids]:
        for rank, doc_id in enumerate(ranked_list, 1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank)
    fused_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:retrieve_k]
    fused_texts = [id_to_text[i] for i in fused_ids]

    # --- cross-encoder reranking ---
    pairs = [(query, t) for t in fused_texts]
    ce_scores = cross_encoder.predict(pairs)
    ranked = sorted(
        zip(fused_ids, fused_texts, [id_to_meta[i] for i in fused_ids], ce_scores),
        key=lambda x: x[3],
        reverse=True,
    )[:final_k]

    return (
        [r[0] for r in ranked],
        [r[1] for r in ranked],
        [r[2] for r in ranked],
        [float(r[3]) for r in ranked],
    )


print("retrieve_with_expansion() defined")

retrieve_with_expansion() defined


In [80]:
# --- Define the three evaluation queries ---
EVAL_QUERIES = [
    {
        "type": "Emissions targets",
        "query": "What are Anglo American's GHG emissions targets for 2020?",
        "expected_facts": ["22%", "30%", "2016", "BAU", "2020", "2030"],
    },
    {
        "type": "Activity of emissions",
        "query": "What was Anglo American's total copper production in 2019?",
        "expected_facts": ["copper", "production", "2019", "tonnes"],
    },
    {
        "type": "Changed over time?",
        "query": "How have Anglo American's GHG emissions changed over time?",
        "expected_facts": ["CO2", "Mt", "emissions", "2019", "2018"],
    },
]

# --- Run retrieval + generation for each query ---
eval_results = []

for i, eq in enumerate(EVAL_QUERIES):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: [{eq['type']}] {eq['query']}")
    print(f"{'='*70}")

    # Retrieve with expansion
    r_ids, r_texts, r_meta, r_scores = retrieve_with_expansion(
        eq["query"], collection, embedding_model, cross_encoder
    )
    print(f"  Retrieved {len(r_ids)} chunks (top CE score: {r_scores[0]:.4f})")

    # Token budget and trimming
    budget = compute_token_budget(
        eq["query"], SYSTEM_MESSAGE_PROD, GEN_WINDOW, gen_tokenizer, MAX_NEW_TOKENS
    )
    trimmed_texts, trimmed_ids, trimmed_meta = trim_chunks_to_budget(
        r_texts, r_ids, r_meta, budget, gen_tokenizer
    )
    print(f"  Chunks after budget trim: {len(trimmed_texts)} (budget: {budget} tokens)")

    # Build prompt and generate
    source_ctx = build_source_context(trimmed_texts, trimmed_ids, trimmed_meta)
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE_PROD},
        {"role": "user", "content": f"{source_ctx}\n\nQuestion: {eq['query']}"},
    ]
    prompt = gen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    raw_out = generator(prompt)
    answer = clean_generated_answer(raw_out[0]["generated_text"])

    print(f"\n  === ANSWER ===")
    print(f"  {answer[:500]}")

    # Check facts
    answer_lower = answer.lower()
    chunks_combined = " ".join(r_texts).lower()
    facts_in_chunks = [f for f in eq["expected_facts"] if f.lower() in chunks_combined]
    facts_in_answer = [f for f in eq["expected_facts"] if f.lower() in answer_lower]

    eval_results.append({
        "type": eq["type"],
        "query": eq["query"],
        "ids": r_ids,
        "texts": r_texts,
        "meta": r_meta,
        "answer": answer,
        "expected_facts": eq["expected_facts"],
        "facts_in_chunks": facts_in_chunks,
        "facts_in_answer": facts_in_answer,
    })

    print(f"\n  Facts in chunks: {facts_in_chunks}")
    print(f"  Facts in answer: {facts_in_answer}")

print(f"\n\nAll {len(EVAL_QUERIES)} queries completed.")


Query 1: [Emissions targets] What are Anglo American's GHG emissions targets for 2020?
  Retrieved 5 chunks (top CE score: 8.4379)
  Chunks after budget trim: 5 (budget: 1524 tokens)

  === ANSWER ===
  According to the given text, Anglo American's GHG emissions targets for 2020 are 16.0 million tonnes of CO2-eq.

  Facts in chunks: ['22%', '30%', '2016', 'BAU', '2020', '2030']
  Facts in answer: ['2020']

Query 2: [Activity of emissions] What was Anglo American's total copper production in 2019?
  Retrieved 5 chunks (top CE score: 5.3126)
  Chunks after budget trim: 4 (budget: 1525 tokens)

  === ANSWER ===
  According to the given data:

Total copper production in 2019 was 248,800 tonnes (Tm).

  Facts in chunks: ['copper', 'production', '2019', 'tonnes']
  Facts in answer: ['copper', 'production', '2019', 'tonnes']

Query 3: [Changed over time?] How have Anglo American's GHG emissions changed over time?
  Retrieved 5 chunks (top CE score: 4.2256)
  Chunks after budget trim: 5 (budg

In [81]:
# --- Build the diagnosis table ---
rows = []
for r in eval_results:
    n_expected = len(r["expected_facts"])
    n_in_chunks = len(r["facts_in_chunks"])
    n_in_answer = len(r["facts_in_answer"])

    # Recall@5: fraction of expected facts found in the top-5 retrieved chunks
    recall_at_5 = n_in_chunks / n_expected if n_expected > 0 else 0.0

    # Determine problem type
    if n_in_chunks == 0:
        problem = "Retrieval"
    elif n_in_answer < n_in_chunks:
        problem = "Generation"
    elif n_in_answer == n_expected:
        problem = "None"
    else:
        problem = "Partial retrieval"

    rows.append({
        "Question": r["type"],
        "Recall@5": f"{recall_at_5:.0%}",
        "Facts in chunks?": f"{n_in_chunks}/{n_expected} ({', '.join(r['facts_in_chunks']) if r['facts_in_chunks'] else 'none'})",
        "Facts in answer?": f"{n_in_answer}/{n_expected} ({', '.join(r['facts_in_answer']) if r['facts_in_answer'] else 'none'})",
        "Problem type": problem,
    })

diagnosis_df = pd.DataFrame(rows)
display(diagnosis_df)

,Question,Recall@5,Facts in chunks?,Facts in answer?,Problem type
0,Emissions targets,100%,"6/6 (22%, 30%, 2016, BAU, 2020, 2030)",1/6 (2020),Generation
1,Activity of emissions,100%,"4/4 (copper, production, 2019, tonnes)","4/4 (copper, production, 2019, tonnes)",None
2,Changed over time?,80%,"4/5 (CO2, Mt, emissions, 2018)","3/5 (CO2, emissions, 2018)",Generation


In [82]:
# --- Inspect full answers and source chunks for each query ---
for i, r in enumerate(eval_results):
    print(f"\n{'='*70}")
    print(f"[{r['type']}] {r['query']}")
    print(f"{'='*70}")
    print(f"\nFull answer:\n{r['answer']}")
    print(f"\nTop-5 chunks:")
    for j, (cid, text, meta) in enumerate(zip(r["ids"], r["texts"], r["meta"])):
        pages = _get_pages(meta)
        source = meta.get("source", "?")
        print(f"  [{j+1}] {cid} ({source}, p.{pages})")
        print(f"      {text[:200]}...")
        print()


[Emissions targets] What are Anglo American's GHG emissions targets for 2020?

Full answer:
According to the given text, Anglo American's GHG emissions targets for 2020 are 16.0 million tonnes of CO2-eq.

Top-5 chunks:
  [1] anglo/AR2018_elem_0184 (anglo/AR2018, p.['34', '35'])
      RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:

HEADER (H2): STRATEGIC REPORT INNOVATION

Our GHG emission-reduction target for 2020 is 22% agains...

  [2] anglo/AR2018_elem_0183 (anglo/AR2018, p.['34', '35'])
      RUNNING TITLE: Anglo American’s climate change policy articulates our commitment to five principles:

HEADER (H2): STRATEGIC REPORT INNOVATION

The Anglo American chief executive and business unit CEO...

  [3] anglo/Anglo American CDP 2020_elem_0123 (anglo/Anglo American CDP 2020, p.['41'])
      RUNNING TITLE: C4. Targets and performance

HEADER (H2): Covered emissions in target year (metric tons CO2e) [auto-calculated] 10953150

Achievi